### 대화 요약 메모리
- ConversaionEntityMemory, ConversationKGMemory 보다 정보의 손실을 최소화하면서 디테일을 보존할 수 있다 : ConversationSummaryMemory

In [ ]:
# ===== 패키지 설치 (최초 1회만 실행) =====
%pip install -q python-dotenv
%pip install -q -U langchain langchain-classic langchain-community langchain-openai langchain-teddynote networkx faiss-cpu

# ===== 필요한 모듈 import =====
import os
from dotenv import load_dotenv
from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, load_prompt
from datetime import datetime

# ===== 환경변수(.env) 로드 =====
load_dotenv()  # override=False가 기본값이므로 시스템 환경변수가 우선순위를 가짐

# (선택) 정상 로드 확인
print("OpenAI 키:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

# ===== LangSmith 추적 시작 =====
logging.langsmith("CH02-Prompt")  # 프로젝트명 입력

# ===== LLM 객체 생성 =====
llm = ChatOpenAI()

In [8]:
from langchain_classic.memory import ConversationSummaryMemory
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()  # 환경변수 로드

memory = ConversationSummaryMemory(
    llm=ChatOpenAI(model_name="gpt-4o", temperature=0), return_messages=True)

In [9]:
memory.save_context(
    inputs={"human": "유럽 여행 패키지의 가격은 얼마인가요?"},
    outputs={"ai": "유럽 14박 15일 패키지의 기본 가격은 3500유로입니다. ..."},
)

memory.save_context(
    inputs={"human": "여행 중에 방문할 주요 관광지는 어디인가요?"},
    outputs={"ai":"이 여행에서는 파리의 에펠탑, 로마의 콜로세움, 베를린의 ..."},
)

memory.save_context(
    inputs={"human": " 여행자 보험은 포함되어 있나요?"},
    outputs={"ai": "네, 모든 여행자에게 기본 여행자 보험을 제공합니다. 이 보험은 ..."},
)

memory.save_context(
    inputs={"human": "항공편 좌석을 비즈니스 클래스로 업그레이드할 수 있나요?"},
    outputs={"ai": "항공편 좌석을 비즈니스 클래스로 업그레이드하는 것이 ..."},
)

memory.save_context(
    inputs={"human": "패키지에 포함된 호텔의 등급은 어떻게 되나요?"},
    outputs={"ai": "이 패키지에는 4성급 호텔 숙박이 포함되어 있습니다. ..."},
)

memory.save_context(
    inputs={"human": "식사 옵션에 대해 더 자세히 알려주실 수 있나요?"},
    outputs={"ai": "이 여행 패키지는 매일 아침 호텔에서 제공되는 조식을 포함하고 ..."},
)


memory.save_context(
    inputs={"human": "패키지 예약 시 예약금은 얼마인가요? 취소 정책은 어떻게 되나요?"},
    outputs={"ai": "패키지 예약 시 500유로의 예약금이 필요합니다. 취소 정책은  ..."},
)

In [10]:
print(memory.load_memory_variables({})["history"])  # 메모리 로드

[SystemMessage(content='The human asks about the price of a European travel package. The AI responds that the basic price for a 14-night, 15-day European package is 3500 euros. The human then asks about the main tourist attractions to be visited during the trip. The AI mentions that the trip includes visits to the Eiffel Tower in Paris, the Colosseum in Rome, and sites in Berlin. The human inquires if travel insurance is included, and the AI confirms that basic travel insurance is provided for all travelers. The human asks if it is possible to upgrade the flight seats to business class, and the AI begins to respond about upgrading to business class. The human then asks about the hotel rating included in the package, and the AI states that the package includes accommodations in 4-star hotels. The human asks for more details about meal options, and the AI explains that the travel package includes daily breakfast provided at the hotel. The human asks about the deposit required for booking

In [11]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryBufferMemory

llm = ChatOpenAI()

memory = ConversationSummaryBufferMemory(
    llm=llm, max_token_limit=200, return_messages=True # 요약의 기준이 되는 토큰 길이를 설정
)

C:\Users\user\AppData\Local\Temp\ipykernel_11500\3616448012.py:6: LangChainDeprecationWarning: The class `ConversationSummaryBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryBufferMemory(


In [12]:
memory.save_context(
    inputs={"human": "유럽 여행 패키지의 가격은 얼마인가요?"},
    outputs={"ai": "유럽 14박 15일 패키지의 기본 가격은 3,500 유로입니다. 이 가격에는 항공료, 호텔 숙박비, 지정된 관광지 입장료가 포함되어 있습니다. 추가 비용은 선택하신 옵션 투어나 개인 경비에 따라 달라집니다."},
)

In [13]:
memory.load_memory_variables({})["history"]  # 메모리 로드

[HumanMessage(content='유럽 여행 패키지의 가격은 얼마인가요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='유럽 14박 15일 패키지의 기본 가격은 3,500 유로입니다. 이 가격에는 항공료, 호텔 숙박비, 지정된 관광지 입장료가 포함되어 있습니다. 추가 비용은 선택하신 옵션 투어나 개인 경비에 따라 달라집니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [14]:
memory.save_context(
    inputs={"human": "여행 중에 방문할 주요 관광지는 어디인가요?"},
    outputs={"ai":"이 여행에서는 파리의 에펠탑, 로마의 콜로세움, 베를린의 브란덴부르크 문 등 유럽의 대표적인 관광지를 방문하게 됩니다. 또한, 각 도시에서 현지 문화와 역사를 체험할 수 있는 다양한 투어와 활동이 포함되어 있습니다. 각 도시의 대표적인 명소들을 포괄적으로 경험하실 수 있습니다."},
)

memory.save_context(
    inputs={"human": " 여행자 보험은 포함되어 있나요?"},
    outputs={"ai": "네, 모든 여행자에게 기본 여행자 보험을 제공합니다. 이 보험은 의료비 지원, 긴급 상황 발생 시 지원 등을 포함합니다. 추가적인 보험 보장을 원하시면 상향 조저이 가능합니다."},
)

memory.save_context(
    inputs={"human": "항공편 좌석을 비즈니스 클래스로 업그레이드할 수 있나요? 비용은 어떻게 되나요?"},
    outputs={"ai": "항공편 좌석을 비즈니스 클래스로 업그레이드하는 것이 가능합니다. 업그레이드 비용은 왕복 기준으로 약 1200유로 추가됩니다. 비즈니스 클래스에서는 더 넓은 좌석, 우수한 기내식, 그리고 추가 수하물 허용량 등의 혜택을 제공합니다."},
)

memory.save_context(
    inputs={"human": "패키지에 포함된 호텔의 등급은 어떻게 되나요?"},
    outputs={"ai": "이 패키지에는 4성급 호텔 숙박이 포함되어 있습니다. 각 호텔은 편안함과 편의성을 제공하며, 중심지에 위치해 관광지와의 접근성이 좋습니다. 모든 호텔은 우수한 서비스와 편의 시설을 갖추고 잇습니다."},
)


In [16]:
memory.load_memory_variables({})["history"]  # 메모리 로드

[SystemMessage(content="The human asks about the price of a European travel package. The AI responds that the basic price for a 14-night, 15-day package is 3,500 euros, which includes airfare, hotel accommodation, and entrance fees to specified attractions. Additional costs may vary depending on optional tours or personal expenses. The AI also mentions that basic traveler's insurance is included for all travelers, covering medical expenses and emergency support. Additional insurance coverage can be requested with higher premiums. The human inquires about upgrading the flight seats to business class and the cost involved. The AI confirms that upgrading to business class is possible with an additional cost of approximately 1200 euros round trip, offering benefits such as wider seats, premium onboard meals, and increased baggage allowance.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='패키지에 포함된 호텔의 등급은 어떻게 되나요?', additional_kwargs={}, response_metadata={}),
 AIMessa

### 벡터 스토어 검색 메모리
- 벡터 스토어 데이텉베이스에 저장하고 조회해 볼 수 있는 기능
- 저장된 과거 대화 기록 중에서 퀴리(조회) 내용과 가장 비슷한 상위 k 개의 문서를 쿼리합니다.
- 시간 순서 고려하지 않고 대화 내용을 검색해서 필요한 내용 뽑음

In [18]:
import faiss
from langchain_openai import OpenAIEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

load_dotenv()

embeddings = OpenAIEmbeddings() # 임베딩 모델을 정의

embedding_size = 1536 # 임베딩 벡터의 차원 수, vector store를 초기화
index = faiss.IndexFlatL2(embedding_size) # L2 distance를 사용하여 벡터를 저장하는 FAISS 인덱스 생성
vectorstore = FAISS(embeddings, index, InMemoryDocstore({}), {}) # FAISS 벡터 스토어 생성

C:\Users\user\AppData\Local\Temp\ipykernel_11500\1708292644.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.in_memory import InMemoryDocstore


In [21]:
from langchain_classic.memory import VectorStoreRetrieverMemory

retriever = vectorstore.as_retriever(search_kwargs={"k": 1})  # 최근 1개의 대화를 검색하도록 설정
memory = VectorStoreRetrieverMemory(retriever=retriever)  # 벡터 검색기 기반 메모리 생성

C:\Users\user\AppData\Local\Temp\ipykernel_11500\226966028.py:4: LangChainDeprecationWarning: The class `VectorStoreRetrieverMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = VectorStoreRetrieverMemory(retriever=retriever)  # 벡터 검색기 기반 메모리 생성


In [22]:
memory.save_context( #임의의 대화를 저장
    inputs={"human": "안녕하세요, 오늘 면접에 ... 자기소개 부탁드립니다."},
    outputs={"ai": "안녕하세요, 저는 컴퓨터 과학을 전공한 신입 개발자입니다. 대학에서는 주로 자바와 파이썬을 사용했으며, 최근에는 웹 개발 프로젝트에 참여하여 실제 사용자를 위한 서비스를 개발하는 경험을 했습니다."},
)

memory.save_context( 
    inputs={"human": "프로젝트에서 어떤 역할을 맡았나요?"},
    outputs={"ai": "제가 맡은 역할은 백엔드 개발자였습니다. 사용자 데이터 처리와 서버 로직개발을 담당했으며, RESTful API를 구현하여 프런트엔드와의 통신을 담당했습니다. 또한, 데이터베이스 설계에도 참여했습니다."},
)

memory.save_context( 
    inputs={"human": "팀프로젝트에서 어려움을 겪었던 경험이 있다면 ..."},
    outputs={"ai": "프로젝트 초기에 의사소통 문제로 몇 가지 어려움이..."},
)

memory.save_context( 
    inputs={"human": "개발자로서의 자신의 강점은 무엇이라고 생각하나요?"},
    outputs={"ai": "제 강점은 빠른 학습 능력과 문제 해결 능력입니다."},
)

In [23]:
print(memory.load_memory_variables({"human":"면접자 전공은 무엇인가요?"})["history"])  # 메모리  로드

human: 안녕하세요, 오늘 면접에 ... 자기소개 부탁드립니다.
ai: 안녕하세요, 저는 컴퓨터 과학을 전공한 신입 개발자입니다. 대학에서는 주로 자바와 파이썬을 사용했으며, 최근에는 웹 개발 프로젝트에 참여하여 실제 사용자를 위한 서비스를 개발하는 경험을 했습니다.


In [26]:
print(memory.load_memory_variables({"human": "면접자가 프로젝트에서 맡은 역할은 무엇인가요?"})["history"])  # 연관성 높은 한 개의 대화를 추출

human: 프로젝트에서 어떤 역할을 맡았나요?
ai: 제가 맡은 역할은 백엔드 개발자였습니다. 사용자 데이터 처리와 서버 로직개발을 담당했으며, RESTful API를 구현하여 프런트엔드와의 통신을 담당했습니다. 또한, 데이터베이스 설계에도 참여했습니다.
